# Scouting process and genuine event-log analysis
Case = one explicitly executed scouting search. Activities: need_defined → search_executed → ranking_generated → shortlist_reviewed → decision_recorded. Only observed app actions are logged. A generated ranking does not imply analyst review or a signing. Unit-test logs are temporary and are not included here.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/config.py').exists():
    if ROOT == ROOT.parent: raise RuntimeError('Run inside the project')
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/tfm-matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/tfm-cache')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from src.config import ROOT, MODELS_DIR
from src.preprocessing.clean import load_players
players = load_players()
dataset = json.loads((ROOT / 'reports/dataset.json').read_text())
display(Markdown('**Dataset:** ' + dataset['dataset_kind'] + '. Legacy-labelled observations are preliminary; no synthetic rows are included.'))
assert not players.source.str.contains('demo|synthetic', case=False).any()

from src.process.events import discover


**Dataset:** real_legacy_unverified. Legacy-labelled observations are preliminary; no synthetic rows are included.

In [2]:
schema=pd.DataFrame([
('case_id','string','Search UUID, not player ID'),
('activity','enum','Observed process step'),
('timestamp','UTC ISO-8601','Actual server event timestamp'),
('actor','string','app_user or system, not an invented named scout'),
('event_kind','enum','observed_application; simulation rejected'),
('attributes','JSON','style, position, season, club filters and candidate count')],columns=['field','type','meaning'])
display(schema)
path=ROOT/'data/events/scouting.jsonl'
try:
    dfg,cases=discover(path)
except ValueError as exc:
    display(Markdown('**Discovery blocked:** '+str(exc)))
else:
    display(dfg);display(cases)
    dfg.to_csv(ROOT/'reports/process_dfg.csv',index=False)
    cases.to_csv(ROOT/'reports/process_cases.csv',index=False)


,field,type,meaning
0,case_id,string,"Search UUID, not player ID"
1,activity,enum,Observed process step
2,timestamp,UTC ISO-8601,Actual server event timestamp
3,actor,string,"app_user or system, not an invented named scout"
4,event_kind,enum,observed_application; simulation rejected
5,attributes,JSON,"style, position, season, club filters and cand..."


**Discovery blocked:** No genuine event log available; process discovery blocked

Directly-follows discovery is available for genuine event logs, with edge counts, median transition time and incomplete-case indicators. There are no genuine human scouting sessions in this submission, so discovery and bottleneck findings are not claimed. Real app usage may create partial observed cases; those are not evidence of actual club recruitment. Formal conformance, completed decisions and organizational KPIs require consented real workflow records. No simulated log is generated to fill the gap.